In [2]:
# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

import sys
import torch
from tqdm import tqdm
from nnsight import LanguageModel

sys.path.append("./mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder
from transformer_lens import utils

torch.set_grad_enabled(False)

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir="./jbloom_dictionaries")

    save_path = f"./jbloom_dictionaries/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

In [3]:
gpt = LanguageModel("openai-community/gpt2", device_map="cuda:0", dispatch=True)

print("Loaded GPT")

Loaded GPT


In [4]:
kwargs = {
    "load_in_4bit": True,
    "device_map": "cuda:0",
    "dispatch": True
}

mixtral = LanguageModel("mistralai/Mixtral-8x7B-Instruct-v0.1", **kwargs)

print("Loaded Mixtral in 4bit")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

Loaded Mixtral in 4bit


In [19]:
with gpt.trace("hello, my name is"):
    activations = gpt.transformer.h[0].output[0]

    _, feature_acts, _, _, _, _ = sae_list[0](activations)
    
    feature_acts.save()

NameError: name 'sae_list' is not defined

In [23]:
import simulator
import importlib 
importlib.reload(simulator)

engine = simulator.RewardEngine(mixtral, [])
agent = simulator.Agent(mixtral, engine)

In [24]:
test = simulator.Feature(
    prompt="abacdqei",
    tokens=["a","b","a","c","d","q", "e","i"], 
    acts=[],
    n_acts=["10","0","10","0","0","0","0",'0']
)

out = agent(test)

print(out)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST] We're studying neurons in a neural network. Each neuron looks for some particular thing in a short document. Look at the parts of the document the neuron activates for and give a detailed explaination about what the neuron is doing. 

The activation format is token<tab>activation. Activation values range from 0 to 10. A neuron finding what it's looking for is represented by a non-zero activation value. The higher the activation value, the stronger the match.

Sentence: 
abacdqei

Activations:
<start>
a   10
b   0
a   10
c   0
d   0
q   0
e   0
i   0
<end>

Here are three concise sentences that best describe the neurons behavior.
 [/INST] 1. The neuron is activated by the letter 'a' and has a strong preference for this character, as indicated by the maximum activation value of 10 in the first and third positions.
2. The neuron does not activate for the letters 'b', 'c', 'd', 'q', 'e', or 'i', as shown by the zero activation values for these characters.
3. The neuron does not 